In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')



In [ ]:
llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [ ]:
class State(TypedDict):
    question:  str
    answer:    str
    score:     int
    round:     int


In [ ]:
def ask_question(state: State) -> State:
    q = llm.invoke(
        f"Give me a fun trivia question (round {state['round']+1}/3). "
        "Just the question, no answer."
    ).content
    print(f"\n❓ {q}")
    answer = input("Your answer: ")
    return {"question": q, "answer": answer}

In [ ]:
def grade_answer(state: State) -> State:
    verdict = llm.invoke(
        f"Q: {state['question']}\nA: {state['answer']}\n"
        "Is this correct? Reply only YES or NO."
    ).content.strip()
    correct = "YES" in verdict.upper()
    print("✅ Correct!" if correct else "❌ Wrong!")
    return {
        "score": state["score"] + (1 if correct else 0),
        "round": state["round"] + 1,
    }

In [ ]:
def should_continue(state: State) -> str:
    return "ask" if state["round"] < 3 else END

In [ ]:
# Build the graph
graph = StateGraph(State)
graph.add_node("ask",   ask_question)
graph.add_node("grade", grade_answer)
graph.set_entry_point("ask")
graph.add_edge("ask", "grade")
graph.add_conditional_edges("grade", should_continue)

app = graph.compile(checkpointer=MemorySaver())
app.invoke(
    {"score": 0, "round": 0, "question": "", "answer": ""},
    config={"configurable": {"thread_id": "game-1"}}
)